# BrewStock Demand Forecast Starter

Notebook ini melengkapi item forecast PRD fix:
- Data sintetik 6 bulan untuk 8 SKU
- Model: Moving Average, SARIMA, Prophet, XGBoost
- Walk-forward validation (time based)
- Tabel komparasi MAPE dan RMSE
- Simulasi dampak bisnis (waste dan stockout)
- Plot actual vs predicted model terbaik
- Export model terbaik ke `ml/forecast/models/model.pkl`


In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 30)



In [ ]:
# dibantu AI: resolveProjectRoot
def resolveProjectRoot() -> Path:
    probePath = Path.cwd().resolve()
    while probePath != probePath.parent:
        if (probePath / 'ml' / 'forecast').exists():
            return probePath
        probePath = probePath.parent
    raise FileNotFoundError('Project root missing')


projectRoot = resolveProjectRoot()
processedDataPath = projectRoot / 'ml' / 'forecast' / 'data' / 'processed' / 'dailyDemandBySku.csv'
modelOutputPath = projectRoot / 'ml' / 'forecast' / 'models' / 'model.pkl'
processedDataPath.parent.mkdir(parents=True, exist_ok=True)
modelOutputPath.parent.mkdir(parents=True, exist_ok=True)

print(f'projectRoot: {projectRoot}')
print(f'processedDataPath: {processedDataPath}')
print(f'modelOutputPath: {modelOutputPath}')



In [ ]:
# dibantu AI: buildHolidayDateSet
def buildHolidayDateSet() -> set[pd.Timestamp]:
    holidayDateItems = pd.to_datetime([
        '2026-01-01',
        '2026-01-26',
        '2026-03-19',
        '2026-03-20',
        '2026-03-21',
        '2026-03-22',
        '2026-05-01',
        '2026-05-14',
    ])
    return {itemValue.normalize() for itemValue in holidayDateItems}


# dibantu AI: buildSyntheticDemandData
def buildSyntheticDemandData() -> pd.DataFrame:
    dateItems = pd.date_range('2026-01-01', '2026-06-30', freq='D')
    skuItems = [
        'arabicaBeans',
        'freshMilk',
        'oatMilk',
        'hazelnutSyrup',
        'vanillaSyrup',
        'chocolateSauce',
        'matchaPowder',
        'cupMedium',
    ]
    baseDemandMap = {
        'arabicaBeans': 46,
        'freshMilk': 62,
        'oatMilk': 25,
        'hazelnutSyrup': 19,
        'vanillaSyrup': 18,
        'chocolateSauce': 22,
        'matchaPowder': 16,
        'cupMedium': 74,
    }

    holidayDateSet = buildHolidayDateSet()
    rowItems = []

    for dateIndex, dateValue in enumerate(dateItems):
        dayOfWeekValue = int(dateValue.dayofweek)
        isWeekendValue = dayOfWeekValue >= 5
        isHolidayValue = dateValue.normalize() in holidayDateSet

        weekendBoostValue = 1.18 if isWeekendValue else 0.98
        holidayBoostValue = 1.34 if isHolidayValue else 1.0
        weeklySeasonValue = 1.0 + 0.15 * np.sin((2.0 * np.pi * dateIndex) / 7.0)
        monthlySeasonValue = 1.0 + 0.08 * np.cos((2.0 * np.pi * dateIndex) / 30.0)
        trendBoostValue = 1.0 + 0.0012 * dateIndex

        for skuIdValue in skuItems:
            baseDemandValue = float(baseDemandMap[skuIdValue])
            noiseMultiplier = float(np.random.normal(1.0, 0.09))
            demandMeanValue = (
                baseDemandValue
                * weekendBoostValue
                * holidayBoostValue
                * weeklySeasonValue
                * monthlySeasonValue
                * trendBoostValue
                * noiseMultiplier
            )
            demandStdValue = max(1.5, demandMeanValue * 0.12)
            demandValue = int(max(0, round(np.random.normal(demandMeanValue, demandStdValue))))

            rowItems.append(
                {
                    'transactionDate': dateValue,
                    'skuId': skuIdValue,
                    'demandQuantity': demandValue,
                    'isWeekend': isWeekendValue,
                    'isHolidaySeason': isHolidayValue,
                }
            )

    demandFrame = pd.DataFrame(rowItems)
    return demandFrame


dailyDemandFrame = buildSyntheticDemandData()
dailyDemandFrame.to_csv(processedDataPath, index=False)

print(f'rows: {len(dailyDemandFrame)}')
print(f'skuCount: {dailyDemandFrame["skuId"].nunique()}')
print(f'dateRange: {dailyDemandFrame["transactionDate"].min().date()} -> {dailyDemandFrame["transactionDate"].max().date()}')
dailyDemandFrame.head()



In [ ]:
# dibantu AI: selectTargetSkuFrame
def selectTargetSkuFrame(dailyDemandFrame: pd.DataFrame) -> tuple[str, pd.DataFrame]:
    skuDemandFrame = (
        dailyDemandFrame.groupby('skuId')['demandQuantity']
        .sum()
        .reset_index()
        .sort_values('demandQuantity', ascending=False)
        .reset_index(drop=True)
    )

    targetSkuId = str(skuDemandFrame.iloc[0]['skuId'])
    targetSkuFrame = (
        dailyDemandFrame[dailyDemandFrame['skuId'] == targetSkuId]
        .copy()
        .sort_values('transactionDate')
        .reset_index(drop=True)
    )

    targetSkuFrame['transactionDate'] = pd.to_datetime(targetSkuFrame['transactionDate'])
    targetSkuFrame = targetSkuFrame[['transactionDate', 'demandQuantity', 'isHolidaySeason']]

    return targetSkuId, targetSkuFrame


targetSkuId, targetSkuFrame = selectTargetSkuFrame(dailyDemandFrame)
holidayDateSet = buildHolidayDateSet()

print(f'targetSkuId: {targetSkuId}')
print(f'targetRows: {len(targetSkuFrame)}')
targetSkuFrame.head()



## Walk-Forward Validation

Split data berdasarkan waktu:
- `minTrainDays = 84`
- `horizonDays = 7`
- `stepDays = 7`

Setiap fold melatih model pada data lampau lalu memprediksi 7 hari ke depan.


In [ ]:
# dibantu AI: calculateMetricValues
def calculateMetricValues(actualArray: np.ndarray, predictedArray: np.ndarray) -> dict[str, float]:
    safeActualArray = np.where(actualArray == 0, 1.0, actualArray)
    absolutePercentErrorArray = np.abs((actualArray - predictedArray) / safeActualArray)
    mapeValue = float(np.mean(absolutePercentErrorArray) * 100.0)
    rmseValue = float(np.sqrt(np.mean((actualArray - predictedArray) ** 2)))
    return {'mapeValue': mapeValue, 'rmseValue': rmseValue}


# dibantu AI: buildHolidayFrame
def buildHolidayFrame(holidayDateSet: set[pd.Timestamp]) -> pd.DataFrame:
    holidayItems = sorted([itemValue.normalize() for itemValue in holidayDateSet])
    holidayFrame = pd.DataFrame({'ds': holidayItems})
    holidayFrame['holiday'] = 'holidaySeason'
    return holidayFrame


# dibantu AI: makeMovingAverageForecast
def makeMovingAverageForecast(trainSeries: pd.Series, horizonDays: int, windowSize: int = 7) -> np.ndarray:
    recentSeries = trainSeries.tail(windowSize)
    averageValue = float(recentSeries.mean()) if len(recentSeries) > 0 else 0.0
    return np.array([averageValue for stepIndex in range(horizonDays)], dtype=float)


# dibantu AI: makeSarimaForecast
def makeSarimaForecast(trainSeries: pd.Series, horizonDays: int) -> np.ndarray:
    modelValue = SARIMAX(trainSeries.astype(float), None, (1, 1, 1), (1, 1, 1, 7))
    fitValue = modelValue.fit(disp=False)
    predictedSeries = fitValue.forecast(steps=horizonDays)
    return np.array(predictedSeries, dtype=float)


# dibantu AI: makeProphetForecast
def makeProphetForecast(trainFrame: pd.DataFrame, horizonDays: int, holidayDateSet: set[pd.Timestamp]) -> np.ndarray:
    prophetTrainFrame = trainFrame[['transactionDate', 'demandQuantity']].rename(
        columns={'transactionDate': 'ds', 'demandQuantity': 'y'}
    )
    prophetModel = Prophet(holidays=buildHolidayFrame(holidayDateSet))
    prophetModel.fit(prophetTrainFrame)
    futureFrame = prophetModel.make_future_dataframe(periods=horizonDays, freq='D')
    predictedFrame = prophetModel.predict(futureFrame)
    predictedArray = np.array(predictedFrame['yhat'].tail(horizonDays), dtype=float)
    return np.array(np.maximum(0.0, predictedArray), dtype=float)


# dibantu AI: createFeatureFrame
def createFeatureFrame(historyFrame: pd.DataFrame, holidayDateSet: set[pd.Timestamp]) -> tuple[pd.DataFrame, np.ndarray]:
    orderedFrame = historyFrame.sort_values('transactionDate').reset_index(drop=True).copy()
    demandItems = orderedFrame['demandQuantity'].astype(float).tolist()
    dateItems = orderedFrame['transactionDate'].tolist()

    featureRowItems = []
    targetItems = []

    for rowIndex in range(7, len(orderedFrame)):
        dateValue = pd.Timestamp(dateItems[rowIndex]).normalize()
        dayOfWeekValue = int(dateValue.dayofweek)
        featureRowItems.append(
            {
                'dayOfWeek': dayOfWeekValue,
                'isWeekend': int(dayOfWeekValue >= 5),
                'isHolidaySeason': int(dateValue in holidayDateSet),
                'lag1': float(demandItems[rowIndex - 1]),
                'lag7': float(demandItems[rowIndex - 7]),
                'rollingMean7': float(np.mean(demandItems[rowIndex - 7 : rowIndex])),
                'trendIndex': int(rowIndex),
            }
        )
        targetItems.append(float(demandItems[rowIndex]))

    featureFrame = pd.DataFrame(featureRowItems)
    targetArray = np.array(targetItems, dtype=float)
    return featureFrame, targetArray


# dibantu AI: fitXgboostModel
def fitXgboostModel(trainFrame: pd.DataFrame, holidayDateSet: set[pd.Timestamp]):
    featureFrame, targetArray = createFeatureFrame(trainFrame, holidayDateSet)
    historyItems = trainFrame['demandQuantity'].astype(float).tolist()

    if len(featureFrame) == 0:
        return None, [], historyItems

    modelValue = XGBRegressor(objective='reg:squarederror')
    modelValue.fit(featureFrame, targetArray)
    featureNameItems = featureFrame.columns.tolist()
    return modelValue, featureNameItems, historyItems


# dibantu AI: makeXgboostForecast
def makeXgboostForecast(trainFrame: pd.DataFrame, testDateItems: list[pd.Timestamp], holidayDateSet: set[pd.Timestamp]) -> np.ndarray:
    modelValue, featureNameItems, historyItems = fitXgboostModel(trainFrame, holidayDateSet)
    horizonDays = len(testDateItems)

    if modelValue is None:
        return makeMovingAverageForecast(trainFrame['demandQuantity'], horizonDays=horizonDays)

    predictedItems = []

    for dateValue in testDateItems:
        currentDate = pd.Timestamp(dateValue).normalize()
        dayOfWeekValue = int(currentDate.dayofweek)
        lag1Value = float(historyItems[-1]) if len(historyItems) >= 1 else 0.0
        lag7Value = float(historyItems[-7]) if len(historyItems) >= 7 else lag1Value

        rollingSourceItems = historyItems[-7:] if len(historyItems) >= 7 else historyItems
        rollingMean7Value = float(np.mean(rollingSourceItems)) if len(rollingSourceItems) > 0 else 0.0

        featureRowFrame = pd.DataFrame(
            [
                {
                    'dayOfWeek': dayOfWeekValue,
                    'isWeekend': int(dayOfWeekValue >= 5),
                    'isHolidaySeason': int(currentDate in holidayDateSet),
                    'lag1': lag1Value,
                    'lag7': lag7Value,
                    'rollingMean7': rollingMean7Value,
                    'trendIndex': int(len(historyItems)),
                }
            ]
        )[featureNameItems]

        predictedValue = float(modelValue.predict(featureRowFrame)[0])
        predictedValue = max(0.0, predictedValue)

        predictedItems.append(predictedValue)
        historyItems.append(predictedValue)

    return np.array(predictedItems, dtype=float)




In [ ]:
# dibantu AI: evaluateModelWithWalkForward
def evaluateModelWithWalkForward(
    modelName: str,
    skuFrame: pd.DataFrame,
    holidayDateSet: set[pd.Timestamp],
    minTrainDays: int = 84,
    horizonDays: int = 7,
    stepDays: int = 7,
) -> dict:
    orderedFrame = skuFrame.sort_values('transactionDate').reset_index(drop=True).copy()

    predictionRowItems = []
    foldMetricItems = []

    foldCounter = 0
    for trainEndIndex in range(minTrainDays, len(orderedFrame) - horizonDays + 1, stepDays):
        foldCounter += 1
        trainFrame = orderedFrame.iloc[:trainEndIndex].copy().reset_index(drop=True)
        testFrame = orderedFrame.iloc[trainEndIndex : trainEndIndex + horizonDays].copy().reset_index(drop=True)

        testDateItems = testFrame['transactionDate'].tolist()

        if modelName == 'movingAverage':
            predictedArray = makeMovingAverageForecast(trainFrame['demandQuantity'], horizonDays=horizonDays)
        elif modelName == 'sarima':
            predictedArray = makeSarimaForecast(trainFrame['demandQuantity'], horizonDays=horizonDays)
        elif modelName == 'prophet':
            predictedArray = makeProphetForecast(trainFrame, horizonDays=horizonDays, holidayDateSet=holidayDateSet)
        elif modelName == 'xgboost':
            predictedArray = makeXgboostForecast(trainFrame, testDateItems=testDateItems, holidayDateSet=holidayDateSet)
        else:
            raise ValueError('Model unknown')

        predictedArray = np.maximum(0.0, predictedArray)
        actualArray = np.array(testFrame['demandQuantity'].astype(float), dtype=float)

        metricMap = calculateMetricValues(actualArray, predictedArray)
        foldMetricItems.append(metricMap)

        for rowIndex in range(len(testFrame)):
            predictionRowItems.append(
                {
                    'modelName': modelName,
                    'foldIndex': foldCounter,
                    'transactionDate': testFrame.loc[rowIndex, 'transactionDate'],
                    'actualDemand': float(actualArray[rowIndex]),
                    'predictedDemand': float(predictedArray[rowIndex]),
                }
            )

    predictionFrame = pd.DataFrame(predictionRowItems)

    if len(foldMetricItems) == 0:
        return {
            'modelName': modelName,
            'mapeValue': float('nan'),
            'rmseValue': float('nan'),
            'predictionFrame': predictionFrame,
            'residualStdValue': 0.0,
            'foldCount': 0,
        }

    mapeValue = float(np.mean([itemValue['mapeValue'] for itemValue in foldMetricItems]))
    rmseValue = float(np.mean([itemValue['rmseValue'] for itemValue in foldMetricItems]))

    residualArray = np.array(predictionFrame['actualDemand'] - predictionFrame['predictedDemand'], dtype=float)
    residualStdValue = float(np.std(residualArray)) if len(residualArray) > 1 else 0.0

    return {
        'modelName': modelName,
        'mapeValue': mapeValue,
        'rmseValue': rmseValue,
        'predictionFrame': predictionFrame,
        'residualStdValue': residualStdValue,
        'foldCount': len(foldMetricItems),
    }


# dibantu AI: runBusinessImpactSimulation
def runBusinessImpactSimulation(
    predictionFrame: pd.DataFrame,
    unitCostValue: float,
    unitPriceValue: float,
    safetyFactor: float = 1.1,
) -> dict[str, float]:
    simulationFrame = predictionFrame.copy()
    simulationFrame['plannedStock'] = np.ceil(simulationFrame['predictedDemand'] * safetyFactor)
    simulationFrame['wasteUnits'] = np.maximum(simulationFrame['plannedStock'] - simulationFrame['actualDemand'], 0.0)
    simulationFrame['stockoutUnits'] = np.maximum(simulationFrame['actualDemand'] - simulationFrame['plannedStock'], 0.0)

    totalWasteUnitsValue = float(simulationFrame['wasteUnits'].sum())
    totalStockoutUnitsValue = float(simulationFrame['stockoutUnits'].sum())

    wasteCostValue = totalWasteUnitsValue * unitCostValue
    lostRevenueValue = totalStockoutUnitsValue * unitPriceValue
    totalRiskValue = wasteCostValue + lostRevenueValue

    return {
        'totalWasteUnits': totalWasteUnitsValue,
        'totalStockoutUnits': totalStockoutUnitsValue,
        'wasteCostValue': wasteCostValue,
        'lostRevenueValue': lostRevenueValue,
        'totalRiskValue': totalRiskValue,
    }



In [ ]:
modelNameItems = ['movingAverage', 'sarima', 'prophet', 'xgboost']
modelResultMap = {}
comparisonRowItems = []

unitCostMap = {
    'arabicaBeans': 22000.0,
    'freshMilk': 12000.0,
    'oatMilk': 15000.0,
    'hazelnutSyrup': 17000.0,
    'vanillaSyrup': 16000.0,
    'chocolateSauce': 15500.0,
    'matchaPowder': 21000.0,
    'cupMedium': 2300.0,
}
unitPriceMap = {
    'arabicaBeans': 48000.0,
    'freshMilk': 43000.0,
    'oatMilk': 47000.0,
    'hazelnutSyrup': 46000.0,
    'vanillaSyrup': 45000.0,
    'chocolateSauce': 44000.0,
    'matchaPowder': 50000.0,
    'cupMedium': 38000.0,
}

unitCostValue = float(unitCostMap.get(targetSkuId, 15000.0))
unitPriceValue = float(unitPriceMap.get(targetSkuId, 45000.0))

for modelName in modelNameItems:
    evaluationMap = evaluateModelWithWalkForward(
        modelName=modelName,
        skuFrame=targetSkuFrame,
        holidayDateSet=holidayDateSet,
        minTrainDays=84,
        horizonDays=7,
        stepDays=7,
    )

    impactMap = runBusinessImpactSimulation(
        predictionFrame=evaluationMap['predictionFrame'],
        unitCostValue=unitCostValue,
        unitPriceValue=unitPriceValue,
        safetyFactor=1.1,
    )

    modelResultMap[modelName] = evaluationMap
    comparisonRowItems.append(
        {
            'modelName': modelName,
            'foldCount': int(evaluationMap['foldCount']),
            'mapeValue': float(evaluationMap['mapeValue']),
            'rmseValue': float(evaluationMap['rmseValue']),
            'totalWasteUnits': float(impactMap['totalWasteUnits']),
            'totalStockoutUnits': float(impactMap['totalStockoutUnits']),
            'wasteCostValue': float(impactMap['wasteCostValue']),
            'lostRevenueValue': float(impactMap['lostRevenueValue']),
            'totalRiskValue': float(impactMap['totalRiskValue']),
        }
    )

comparisonFrame = pd.DataFrame(comparisonRowItems).sort_values(['mapeValue', 'rmseValue']).reset_index(drop=True)
comparisonFrame



## Comparison Table

Tabel menampilkan akurasi dan simulasi risiko bisnis per model.
Semakin kecil `mapeValue`, `rmseValue`, dan `totalRiskValue`, semakin baik.


In [ ]:
comparisonDisplayFrame = comparisonFrame.copy()
for columnName in ['mapeValue', 'rmseValue', 'totalWasteUnits', 'totalStockoutUnits']:
    comparisonDisplayFrame[columnName] = comparisonDisplayFrame[columnName].round(3)
for columnName in ['wasteCostValue', 'lostRevenueValue', 'totalRiskValue']:
    comparisonDisplayFrame[columnName] = comparisonDisplayFrame[columnName].round(2)
comparisonDisplayFrame



In [ ]:
bestModelName = str(comparisonFrame.iloc[0]['modelName'])
bestPredictionFrame = modelResultMap[bestModelName]['predictionFrame'].copy()
lastFoldValue = int(bestPredictionFrame['foldIndex'].max())
plotFrame = bestPredictionFrame[bestPredictionFrame['foldIndex'] == lastFoldValue].copy().sort_values('transactionDate')

residualStdValue = float(modelResultMap[bestModelName]['residualStdValue'])
plotFrame['lowerBound'] = np.maximum(0.0, plotFrame['predictedDemand'] - (1.96 * residualStdValue))
plotFrame['upperBound'] = plotFrame['predictedDemand'] + (1.96 * residualStdValue)

plt.figure(figsize=(12, 5))
plt.plot(plotFrame['transactionDate'], plotFrame['actualDemand'], label='Actual', marker='o')
plt.plot(plotFrame['transactionDate'], plotFrame['predictedDemand'], label='Predicted', marker='o')
plt.fill_between(
    plotFrame['transactionDate'],
    plotFrame['lowerBound'],
    plotFrame['upperBound'],
    alpha=0.2,
    label='Confidence Interval',
)
plt.title(f'Best Model: {bestModelName} | SKU: {targetSkuId} | Last Fold')
plt.xlabel('Date')
plt.ylabel('Demand')
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
baselineRiskValue = float(comparisonFrame.loc[comparisonFrame['modelName'] == 'movingAverage', 'totalRiskValue'].iloc[0])
bestRiskValue = float(comparisonFrame.iloc[0]['totalRiskValue'])
riskReductionValue = baselineRiskValue - bestRiskValue

businessImpactFrame = pd.DataFrame(
    [
        {
            'targetSkuId': targetSkuId,
            'baselineModel': 'movingAverage',
            'bestModel': bestModelName,
            'baselineRiskValue': baselineRiskValue,
            'bestRiskValue': bestRiskValue,
            'riskReductionValue': riskReductionValue,
        }
    ]
)

businessImpactFrame



In [ ]:
# dibantu AI: trainBestModelForExport
def trainBestModelForExport(
    modelName: str,
    skuFrame: pd.DataFrame,
    holidayDateSet: set[pd.Timestamp],
    horizonDays: int = 7,
) -> tuple[dict, pd.DataFrame]:
    trainFrame = skuFrame.sort_values('transactionDate').reset_index(drop=True).copy()

    lastDateValue = pd.Timestamp(trainFrame['transactionDate'].iloc[-1]).normalize()
    futureDateItems = list(pd.date_range(lastDateValue + pd.Timedelta(days=1), periods=horizonDays, freq='D'))

    if modelName == 'movingAverage':
        futureForecastArray = makeMovingAverageForecast(trainFrame['demandQuantity'], horizonDays=horizonDays)
        artifactMap = {
            'modelName': modelName,
            'windowSize': 7,
            'skuId': targetSkuId,
            'lastWindowDemandItems': trainFrame['demandQuantity'].tail(7).astype(float).tolist(),
        }
    elif modelName == 'sarima':
        modelValue = SARIMAX(trainFrame['demandQuantity'].astype(float), None, (1, 1, 1), (1, 1, 1, 7))
        fitValue = modelValue.fit(disp=False)
        futureForecastArray = np.array(fitValue.forecast(steps=horizonDays), dtype=float)
        artifactMap = {
            'modelName': modelName,
            'skuId': targetSkuId,
            'fitModel': fitValue,
        }
    elif modelName == 'prophet':
        prophetTrainFrame = trainFrame[['transactionDate', 'demandQuantity']].rename(
            columns={'transactionDate': 'ds', 'demandQuantity': 'y'}
        )
        prophetModel = Prophet(holidays=buildHolidayFrame(holidayDateSet))
        prophetModel.fit(prophetTrainFrame)
        futureFrame = prophetModel.make_future_dataframe(periods=horizonDays, freq='D')
        predictedFrame = prophetModel.predict(futureFrame)
        futureForecastArray = np.array(predictedFrame['yhat'].tail(horizonDays), dtype=float)
        artifactMap = {
            'modelName': modelName,
            'skuId': targetSkuId,
            'fitModel': prophetModel,
            'holidayDateItems': sorted([itemValue.isoformat() for itemValue in holidayDateSet]),
        }
    elif modelName == 'xgboost':
        modelValue, featureNameItems, historyItems = fitXgboostModel(trainFrame, holidayDateSet)
        if modelValue is None:
            futureForecastArray = makeMovingAverageForecast(trainFrame['demandQuantity'], horizonDays=horizonDays)
        else:
            futureForecastArray = makeXgboostForecast(trainFrame, testDateItems=futureDateItems, holidayDateSet=holidayDateSet)

        artifactMap = {
            'modelName': modelName,
            'skuId': targetSkuId,
            'fitModel': modelValue,
            'featureNameItems': featureNameItems,
            'historyItems': historyItems,
            'holidayDateItems': sorted([itemValue.isoformat() for itemValue in holidayDateSet]),
        }
    else:
        raise ValueError('Model unknown')

    futureForecastArray = np.maximum(0.0, np.array(futureForecastArray, dtype=float))
    futureForecastFrame = pd.DataFrame(
        {
            'transactionDate': futureDateItems,
            'predictedDemand': futureForecastArray,
        }
    )

    return artifactMap, futureForecastFrame


bestArtifactMap, futureForecastFrame = trainBestModelForExport(
    modelName=bestModelName,
    skuFrame=targetSkuFrame,
    holidayDateSet=holidayDateSet,
    horizonDays=7,
)

joblib.dump(bestArtifactMap, modelOutputPath)
print(f'exportPath: {modelOutputPath}')
futureForecastFrame




## Output Checklist

Jika semua sel sudah berjalan:
- File data sintetik ada di `ml/forecast/data/processed/dailyDemandBySku.csv`
- Tabel perbandingan 4 model muncul
- Simulasi dampak bisnis muncul
- Plot model terbaik muncul
- Model terbaik diexport ke `ml/forecast/models/model.pkl`
